# exp083_pf_beam_true_tvt_2d_well_eda train

PF/Beam true-TVT and ANCC/Z physical decomposition EDA using the exp072 feature cache consumed by exp073.

## Contents

1. Setup and configuration
2. Source and schema check
3. Well-level EDA plots
4. Metrics and generated outputs

## 1. Setup and configuration

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

from settings import EXPERIMENT_NAME, ExperimentPaths, get_nested, load_config
from pf_beam_true_tvt_eda import build_source_spec, run_eda, to_jsonable

DEBUG = os.environ.get("EXPERIMENT_DEBUG", "0") == "1"
MAX_PLOTS_ENV = os.environ.get("EXPERIMENT_MAX_PLOTS")
MAX_PLOTS = int(MAX_PLOTS_ENV) if MAX_PLOTS_ENV else None

paths = ExperimentPaths()
paths.require_kaggle_runtime()
paths.ensure_output_dirs()
config = load_config()

print("Experiment:", EXPERIMENT_NAME)
print("Route:", get_nested(config, "experiment.route"))
print("Anchor:", get_nested(config, "lineage.anchor"))
print("Parent input cache:", get_nested(config, "lineage.parent"))
print("EDA mode:", get_nested(config, "eda.mode"))
print("Physical decomposition:", get_nested(config, "eda.physical_decomposition.enabled"))
print("Physical background:", get_nested(config, "eda.physical_background.enabled"))
print("Derivative panel:", get_nested(config, "eda.derivative_panel.enabled"))
print("Debug:", DEBUG)
print("Max plots override:", MAX_PLOTS)

## 2. Source and schema check

In [ ]:
source = build_source_spec(config, paths.root)
print("Source name:", source.name)
print("Source path:", source.path)
print("Source exists:", source.path.exists())
print("Well column:", source.well_column)
print("Target column:", source.target_column)
print("Target delta column:", source.target_delta_column)
print("Base column:", source.base_column)
print("X column:", source.x_column)
print("Candidate columns:")
for item in source.candidate_columns:
    print(" -", item)
print("Physical decomposition config:", get_nested(config, "eda.physical_decomposition"))
print("Physical background config:", get_nested(config, "eda.physical_background"))
print("Derivative panel config:", get_nested(config, "eda.derivative_panel"))

if not source.path.exists():
    raise FileNotFoundError(source.path)

## 3. Well-level EDA plots

In [ ]:
summary = run_eda(config=config, paths=paths, debug=DEBUG, max_plots=MAX_PLOTS)
print(json.dumps(to_jsonable(summary), indent=2, sort_keys=True))

## 4. Metrics and generated outputs

In [ ]:
output_paths = summary["outputs"]
for key, value in output_paths.items():
    print(f"{key}: {value}")

manifest_path = Path(output_paths["plot_manifest_csv"])
well_summary_path = Path(output_paths["well_summary_csv"])
print("metrics.json:", paths.metrics_path, paths.metrics_path.exists())
print("plot manifest exists:", manifest_path.exists())
print("well summary exists:", well_summary_path.exists())